In [1]:
from pynq import Overlay
import numpy as np
from pynq import Xlnk
import struct
from scipy.misc import imread
import cv2
from pynq import Clocks
from PIL import Image
import select
import socket
import cv2
import threading
import struct
import time
import numpy as np
import matplotlib.pyplot as plt
xlnk = Xlnk()
ol=Overlay("size3/base.bit")
ol.download()
print(ol.ip_dict.keys())
net=ol.Conv3_0
print(f'FCLK0: {Clocks.fclk0_mhz:.6f}MHz')

dict_keys(['Conv3_0'])
FCLK0: 187.500000MHz


In [2]:
def RunNet(net,input,output,cache,mode):
    net.write(0x10,input.physical_address)
    net.write(0x18,output.physical_address)
    net.write(0x20,cache.physical_address)
    net.write(0x28,mode)
    net.write(0, (net.read(0)&0x80)|0x01 )
    tp=net.read(0)
    while not ((tp>>1)&0x1):
        tp=net.read(0)
        
WIDTH = 320
HEIGHT = 240
DIM = 3
input1= xlnk.cma_array(shape=(DIM, HEIGHT,WIDTH), cacheable=True, dtype=np.float32)
input2= xlnk.cma_array(shape=(DIM, HEIGHT,WIDTH), cacheable=True, dtype=np.float32)
input11= xlnk.cma_array(shape=(DIM, HEIGHT,WIDTH), cacheable=True, dtype=np.float32)
input22= xlnk.cma_array(shape=(DIM, HEIGHT,WIDTH), cacheable=True, dtype=np.float32)
cache1= xlnk.cma_array(shape=(DIM, HEIGHT, WIDTH), cacheable=True, dtype=np.float32)
cache2= xlnk.cma_array(shape=(DIM,HEIGHT, WIDTH), cacheable=True, dtype=np.float32)
cache11= xlnk.cma_array(shape=(DIM, HEIGHT, WIDTH), cacheable=True, dtype=np.float32)
cache22= xlnk.cma_array(shape=(DIM,HEIGHT, WIDTH), cacheable=True, dtype=np.float32)
output= xlnk.cma_array(shape=(DIM, HEIGHT, WIDTH), cacheable=True, dtype=np.float32)
image_hwc = np.zeros_like(output)

In [3]:
import time 

def get_time():
# 格式化为小时:分钟:秒.毫秒
    formatted_time = time.strftime("%M:%S", time.localtime(time.time())) + f".{int((time.time() % 1) * 1000):03d}"
    return formatted_time

In [4]:
FIXED_DATA_LENGTH = 921600  # 固定数据长度
TOTAL_PACKET_LENGTH = 2 + FIXED_DATA_LENGTH + 1  
class Camera_Connect_Object:
    def __init__(self, D_addr_port=["", 9999]):
        self.addr_port = D_addr_port
        self.is_thread_running = False
        self.status = 0
        self.buffer = None
        self.imageBuffer = None
        self.imageBuffer_ = None
        self.func = b'\x00'
        self.len = 0
        self.input_id = 1
        self.ifOccupySend = False
        self.ifProcessReady = False
        self.isUseCache1 = False
        self.isUseCache2 = False
        self.netStatus = 0
        self.EN_PIC = False
        self.isUseImageBufferCache = False
        self.isStartStorageImage = False
        self.data = bytes()
        self.a = time.time()
        self.time1=None
        self.time11=None
        self.time2=None
        self.time22=None
        self.isLoad1 = True
        self.isLoad2 = True
        self.number = 0
        
    def Set_socket(self):
        # 设置客户端和服务器端套接字
        self.client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.client.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.client.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 1024 * 1024 * 4)  # 设置接收缓冲区为 1 MB
        self.client.setsockopt(socket.SOL_SOCKET, socket.SO_SNDBUF, 1024 * 1024)  # 设置发送缓冲区为 1 MB
    def Socket_Connect(self):
        self.Set_socket()
        self.client.connect(self.addr_port)  # 连接到目标地址和端口
    def Process_Image(self):
        while True:
            while not self.EN_PIC:
                pass
            a = time.time()
            if self.input_id == 1:
                if self.isUseCache2:
                    print("使用input22",self.time22)
                    RunNet(net, input2, cache1, cache2, 1)
                    self.netStatus = 1
                    self.ifProcessReady = True
                    RunNet(net, cache1, cache2, cache1, 3)
                    self.netStatus = 2
                    self.ifProcessReady = True
                    RunNet(net, cache2, output, input22, 2)
                    self.netStatus = 3
                else:
                    print("使用input2",self.time2)
                    RunNet(net, input2, cache1, cache2, 1)
                    self.netStatus = 1
                    self.ifProcessReady = True
                    RunNet(net, cache1, cache2, cache1, 3)
                    self.netStatus = 2
                    self.ifProcessReady = True
                    RunNet(net, cache2, output, input2, 2)
                    self.netStatus = 3
                self.ifProcessReady = True
                while not self.isLoad1:
                    pass
                self.isLoad1 = False
                self.input_id = 2
            elif self.input_id == 2:
                if self.isUseCache1:
                    print("使用input11",self.time11)
                    RunNet(net, input11, cache1, cache2, 1)
                    self.netStatus = 1
                    self.ifProcessReady = True
                    RunNet(net, cache1, cache2, cache1, 3)
                    self.netStatus = 2
                    self.ifProcessReady = True
                    RunNet(net, cache2, output, input11, 2)
                    self.netStatus = 3
                else:
                    print("使用input1",self.time1)
                    RunNet(net, input1, cache1, cache2, 1)
                    self.netStatus = 1
                    self.ifProcessReady = True
                    RunNet(net, cache1, cache2, cache1, 3)
                    self.netStatus = 2
                    self.ifProcessReady = True
                    RunNet(net, cache2, output, input1, 2)
                    self.netStatus = 3
                self.ifProcessReady = True
                while not self.isLoad2:
                    pass
                self.isLoad2 = False
                self.input_id = 1
    def Send_Image(self):
        while True:
            while not self.ifProcessReady:
                pass
            self.ifProcessReady = False
            """if self.netStatus == 3:
                image_hwc = np.copy(output)
                img_data = image_hwc.tobytes()
                self.Send_Data(0x03,len(img_data),img_data)"""
            if self.netStatus == 1:
                image_hwc = np.copy(cache1)
                img_data = image_hwc.tobytes()
                self.Send_Data(0x03,len(img_data),img_data)
            elif self.netStatus == 2:
                image_hwc = np.copy(cache2)
                img_data = image_hwc.tobytes()
                self.Send_Data(0x04,len(img_data),img_data)
            elif self.netStatus == 3:
                image_hwc = np.copy(output)
                img_data = image_hwc.tobytes()
                self.Send_Data(0x05,len(img_data),img_data)
                
    def RT_Image(self):
        self.Start_Receive_Image()
        while True:
            if self.status == 0:
                data = self.client.recv(1)
                if data:
                    if data == b'\xAA':
                        self.status = 1
                else:
                    pass
            elif self.status == 1:
                self.func = self.client.recv(1)
                data_length_bytes = self.client.recv(4)
                self.a = time.time()
                while len(data_length_bytes) < 4:
                    data_length_bytes += self.client.recv(4 - len(data_length_bytes))
                self.len = struct.unpack("l", data_length_bytes)[0]
                self.buffer = self.client.recv(self.len)
                print("接收时间",time.time()-self.a)
                while len(self.buffer) < self.len:
                    self.buffer += self.client.recv(self.len - len(self.buffer))
                vaild = self.client.recv(1)
                if vaild == b'\xAF':
                    self.Deal_With_Data()
                self.buffer = None
                self.func = b'\x00'
                self.len = 0
                self.status = 0
                    
    def Deal_With_Data(self):
        if(self.func == b'\x01'):#Start!
            self.Start_Receive_Image()
        elif(self.func == b'\x02'):
            if self.isUseImageBufferCache:
                self.imageBuffer = bytes(self.buffer)  # 深复制
                self.isUseImageBufferCache = False
            else:
                self.imageBuffer_ = bytes(self.buffer)  # 深复制
                self.isUseImageBufferCache = True
            self.isStartStorageImage = True
            
    def Storage_Image(self):
        while True:
            while not self.isStartStorageImage:
                pass
            if self.isUseImageBufferCache:
                image_data = np.frombuffer(self.imageBuffer_, dtype=np.float32)
            else:
                image_data = np.frombuffer(self.imageBuffer, dtype=np.float32)
            image_data = image_data.reshape((DIM, HEIGHT, WIDTH))
            
            if self.input_id == 1:
                if self.isUseCache1:
                    np.copyto(input1, image_data)
                    self.isUseCache1 = False
                    self.time1 = get_time()
                    print("更新1")
                else:
                    np.copyto(input11, image_data)
                    self.isUseCache1 = True
                    self.time11 = get_time()
                    print("更新11")
                self.isLoad1 = True
            elif self.input_id == 2:
                if self.isUseCache2:
                    np.copyto(input2, image_data)
                    self.isUseCache2 = False
                    self.time2 = get_time()
                    print("更新2")
                else:
                    np.copyto(input22, image_data)
                    self.isUseCache2 = True
                    self.time22 = get_time()
                    print("更新22")
                self.isLoad2 = True
            self.Send_Data(0x02,1,struct.pack("B", 0x02))
            self.isStartStorageImage = False
            self.EN_PIC = True
        
        
    def Start_Receive_Image(self):
        print("请求服务器启动!")
        self.Send_Data(0x01,1,struct.pack("B", 0x01))
        
        
    def Send_Data(self,func,length,data):
        while self.ifOccupySend :
            pass
        self.ifOccupySend = True
        self.client.send(struct.pack("B", 0xAA))
        self.client.send(struct.pack("B", func))
        self.client.send(struct.pack("l", length))
        self.client.send(data)
        self.client.send(struct.pack("B", 0xAF))
        self.ifOccupySend = False
        
    def Get_Data(self):
        # 启动一个线程接收实时图像数据
        threading.Thread(target=self.RT_Image).start()
        threading.Thread(target=self.Process_Image).start()
        threading.Thread(target=self.Send_Image).start()
        threading.Thread(target=self.Storage_Image).start()

In [5]:

if __name__ == '__main__':
    camera=Camera_Connect_Object()
    camera.addr_port[0]="192.168.2.1"
    camera.addr_port=tuple(camera.addr_port)
    camera.Socket_Connect()
    camera.Get_Data()

请求服务器启动!
接收时间 0.0016875267028808594
更新11
使用input2 None
接收时间 0.0062351226806640625
使用input11 23:18.681
更新22
使用input22 23:18.962
接收时间 0.001356363296508789
更新1
使用input1 23:19.286
接收时间 0.017055749893188477
更新2
使用input2 23:19.687
接收时间 0.0005965232849121094
更新11
使用input11 23:20.100
接收时间 0.0016245841979980469
更新22
使用input22 23:20.520
接收时间 0.0059201717376708984
更新1
使用input1 23:20.907
接收时间 0.017182111740112305
更新2
使用input2 23:21.293
接收时间 0.0006725788116455078
更新11
使用input11 23:21.722
接收时间 0.005896091461181641
更新22
使用input22 23:22.135
接收时间 0.005471229553222656
更新1
使用input1 23:22.647
接收时间 0.005669355392456055
更新2
使用input2 23:22.980
接收时间 0.00571894645690918
更新11
使用input11 23:23.443
接收时间 0.0013949871063232422
更新22
使用input22 23:23.961
接收时间 0.0008258819580078125
更新1
使用input1 23:24.370
接收时间 0.0057523250579833984
更新2
使用input2 23:24.787
接收时间 0.011682748794555664
更新11
使用input11 23:25.142
接收时间 0.006790637969970703
更新22
使用input22 23:25.652
接收时间 0.0057525634765625
更新1
使用input1 23:26.007
接收时间 0.0056738853454

更新1
使用input1 24:38.187
接收时间 0.0002448558807373047
更新2
使用input2 24:38.629
接收时间 0.0005273818969726562
更新11
使用input11 24:39.155
接收时间 0.0005338191986083984
更新22
使用input22 24:39.643
接收时间 0.0012977123260498047
更新1
使用input1 24:40.106
接收时间 0.009502172470092773
更新2
使用input2 24:40.524
接收时间 0.005310773849487305
更新11
使用input11 24:41.081
接收时间 0.010648012161254883
更新22
使用input22 24:41.464
接收时间 0.0055141448974609375
更新1
使用input1 24:41.835
接收时间 0.005748748779296875
更新2
使用input2 24:42.252
接收时间 0.005388975143432617
更新11
使用input11 24:42.773
接收时间 0.010994195938110352
更新22
使用input22 24:43.096
接收时间 0.006186723709106445
更新1
使用input1 24:43.450
接收时间 0.006009340286254883
更新2
使用input2 24:44.054
接收时间 0.005604982376098633
更新11
使用input11 24:44.413
接收时间 0.005879402160644531
更新22
使用input22 24:44.772
接收时间 0.005410671234130859
更新1
使用input1 24:45.197
接收时间 0.005469560623168945
更新2
使用input2 24:45.633
接收时间 0.02048635482788086
更新11
使用input11 24:46.138
接收时间 0.005480766296386719
更新22
使用input22 24:46.455
接收时间 0.015432834625244

接收时间 0.0011034011840820312
更新1
使用input1 26:14.063
接收时间 0.005963325500488281
更新2
使用input2 26:14.754
接收时间 0.005993843078613281
更新11
使用input11 26:15.246
接收时间 0.005980491638183594
更新22
使用input22 26:16.263
接收时间 0.0009918212890625
更新1
使用input1 26:17.084
接收时间 0.0007786750793457031
更新2
使用input2 26:17.792
接收时间 0.000885009765625
更新11
使用input11 26:18.433
接收时间 0.0059583187103271484
更新22
使用input22 26:19.017
接收时间 0.0008373260498046875
更新1
使用input1 26:19.670
接收时间 0.005990028381347656
更新2
使用input2 26:20.499
接收时间 0.0007824897766113281
更新11
使用input11 26:21.180
接收时间 0.0008478164672851562
更新22
使用input22 26:21.771
接收时间 0.005845785140991211
更新1
使用input1 26:22.321
接收时间 0.051978349685668945
更新2
使用input2 26:23.051
接收时间 0.026404142379760742
更新11
使用input11 26:23.792
接收时间 0.005832195281982422
更新22
使用input22 26:24.658
接收时间 0.016226768493652344
更新1
使用input1 26:25.187
接收时间 0.03156614303588867
更新2
使用input2 26:25.822
接收时间 0.006122112274169922
更新11
使用input11 26:26.444
接收时间 0.005963563919067383
更新22
使用input22 26:27.150


接收时间 0.005979061126708984
更新1
使用input1 28:08.697
接收时间 0.016187667846679688
更新2
使用input2 28:09.488
接收时间 0.00618290901184082
更新11
使用input11 28:10.233
接收时间 0.02651381492614746
更新22
使用input22 28:10.820
接收时间 0.01619887351989746
更新1
使用input1 28:11.479
接收时间 0.00592350959777832
更新2
使用input2 28:12.075
接收时间 0.0007989406585693359
更新11
使用input11 28:13.202
接收时间 0.0007753372192382812
更新22
使用input22 28:14.500
接收时间 0.026458740234375
更新1
使用input1 28:15.334
接收时间 0.005886077880859375
更新2
使用input2 28:16.182
接收时间 0.005921602249145508
更新11
使用input11 28:16.817
接收时间 0.0059757232666015625
更新22
使用input22 28:17.411
接收时间 0.0008151531219482422
更新1
使用input1 28:18.154
接收时间 0.04393362998962402
更新2
使用input2 28:18.653
接收时间 0.005985260009765625
更新11
使用input11 28:19.230
接收时间 0.005984306335449219
更新22
使用input22 28:19.904


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 假设 x 的形状为 (3, 480, 640)
# 转换为 HWC 格式
x_hwc = np.transpose(output, (1, 2, 0))
print(output)
# 如果图像是 BGR 格式，使用以下代码转换为 RGB
# x_rgb = cv2.cvtColor(x_hwc, cv2.COLOR_BGR2RGB)

# 显示图像
plt.imshow(x_hwc)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_hwc = np.transpose(input, (1, 2, 0))*255

# 如果图像是 BGR 格式，使用以下代码转换为 RGB

# x_rgb = cv2.cvtColor(x_hwc, cv2.COLOR_BGR2RGB)

# 显示图像
plt.imshow(x_hwc)
plt.show()

In [ ]:

# 读取图像
img = Image.open("99.png").convert("RGB")

# 将图像调整为指定大小 (640, 480)
img = img.resize((320, 240))

# 将图像转换为 NumPy 数组并调整通道顺序
x_ori = np.array(img).transpose((2, 0, 1)) /255 # 转换为 (C, H, W)
np.copyto(input1, x_ori)
print(input)

In [ ]:
a = time.time()
RunNet(net, input1, cache1, cache2, 1)
RunNet(net, cache1, cache2, cache1, 3)
RunNet(net, cache2, output, input1, 2)
print(time.time() - a)

In [ ]:

RunNet(net, input1, cache1, cache2, 1)
RunNet(net, cache1, cache2, cache1, 3)
RunNet(net, cache2, output, input1, 2)


In [ ]:

np.copyto(input1, image_data)


In [ ]:
img = Image.open("99.png").convert("RGB")

# 将图像调整为指定大小 (640, 480)
img = img.resize((320, 240))

# 将图像转换为 NumPy 数组并调整通道顺序
x_ori= np.array(img).transpose((2, 0, 1)) / 255 # 转换为 (C, H, W)
a=time.time()
np.copyto(input1, x_ori)
print(time.time()-a)